# Herring Spawn: Initial Scene Search

Search Sentinel-2 for known spawn events, export thumbnails, compute spectral features, and generate a review page.

In [ ]:
import json, sys
from pathlib import Path
from datetime import date, timedelta
from shapely.geometry import shape
import ee

ee.Initialize(project="redd-fish")
sys.path.insert(0, str(Path.cwd()))

from herring_spawner.imagery.gee import GeeSentinel2Provider
from herring_spawner.imagery.base import SearchRequest
from herring_spawner.features.spectral import compute_visible_features
from herring_spawner.review.static import write_review_page
from herring_spawner.models import Event, LabelConfidence, SourceType

In [ ]:
# Load events with known dates
payload = json.loads(Path("data/interim/events.geojson").read_text())
events = []
for feat in payload["features"]:
    p = feat["properties"]
    if not p.get("start_date"):
        continue
    events.append({
        "id": p["event_id"],
        "start": date.fromisoformat(p["start_date"]),
        "end": date.fromisoformat(p["end_date"]) if p.get("end_date") else date.fromisoformat(p["start_date"]),
        "geom": shape(feat["geometry"]),
        "source": p["source_type"],
    })
print(f"Loaded {len(events)} events with known dates")
for e in events:
    print(f"  {e['id']}: {e['start']} ({e['source']})")

In [ ]:
# Search Sentinel-2 for each event
provider = GeeSentinel2Provider()
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
results = []

for event in events:
    start = (event["start"] - timedelta(days=10)).isoformat()
    end = (event["end"] + timedelta(days=10)).isoformat()
    minx, miny, maxx, maxy = event["geom"].buffer(0.02).bounds
    
    scenes = (
        collection
        .filterBounds(ee.Geometry.Rectangle(minx, miny, maxx, maxy))
        .filterDate(start, end)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 80))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    
    info = scenes.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()
    ids = scenes.aggregate_array("system:index").getInfo()
    dates = scenes.aggregate_array("system:time_start").getInfo()
    
    for i, (sid, cloud, ts) in enumerate(zip(ids, info, dates)):
        import datetime
        dt = datetime.datetime.utcfromtimestamp(ts / 1000.0)
        results.append({
            "event_id": event["id"],
            "scene_id": sid,
            "date": dt.strftime("%Y-%m-%d"),
            "cloud": cloud,
            "bounds": (minx, miny, maxx, maxy),
        })
    print(f"{event['id']}: {len(ids)} scenes (cloud < 80%)")

print(f"\nTotal: {len(results)} scene candidates")
print(f"\nClear scenes (cloud < 20%):")
for r in results:
    if r["cloud"] < 20:
        print(f"  {r['event_id']}: {r['scene_id']} ({r['date']}, cloud={r['cloud']}%)")

In [ ]:
# Export RGB thumbnails for each clear scene
Path("data/exports").mkdir(parents=True, exist_ok=True)
Path("data/review").mkdir(parents=True, exist_ok=True)
thumbnails = []

for r in results:
    if r["cloud"] >= 20:
        continue
    minx, miny, maxx, maxy = r["bounds"]
    scene = ee.Image(f"{provider.collection}/{r['scene_id']}")
    rgb = scene.select(["B4", "B3", "B2"])
    
    thumbnail_url = rgb.getThumbURL({
        "min": 0,
        "max": 3000,
        "region": ee.Geometry.Rectangle(minx, miny, maxx, maxy),
        "dimensions": 512,
        "format": "png",
    })
    
    import requests
    thumb_path = f"data/review/thumb_{r['event_id']}_{r['scene_id']}.png"
    resp = requests.get(thumbnail_url)
    with open(thumb_path, "wb") as f:
        f.write(resp.content)
    
    thumbnails.append({
        "event_id": r["event_id"],
        "scene_id": r["scene_id"],
        "date": r["date"],
        "thumbnail": thumb_path,
    })
    print(f"Downloaded: {thumb_path}")

print(f"\n{len(thumbnails)} thumbnails exported")

In [ ]:
# Show results in a table
from IPython.display import display, HTML

if thumbnails:
    rows_html = ""
    for t in thumbnails:
        import base64
        with open(t["thumbnail"], "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        rows_html += f"""
        <tr>
            <td>{t['event_id']}</td>
            <td>{t['scene_id']}</td>
            <td>{t['date']}</td>
            <td><img src="data:image/png;base64,{b64}" width="256"></td>
        </tr>
        """
    display(HTML(f"""
        <table border="1">
            <tr><th>Event</th><th>Scene</th><th>Date</th><th>Thumbnail</th></tr>
            {rows_html}
        </table>
    """))
else:
    print("No thumbnails to display.")

## Next Steps

1. Open `data/review/review.html` to visually inspect thumbnails
2. Label each chip: `likely_spawn`, `possible_spawn`, `no_spawn`, `cloud_blocked`
3. Run Clay embeddings on usable chips
4. Run similarity search against the known-spawn examples